# Entrenamiento del modelo clasificador de riesgo IA

En este notebook vamos a entrenar un modelo baseline para clasificar textos en 4 categorías de riesgo del AI Act:
- Inaceptable
- Alto riesgo
- Riesgo limitado
- Riesgo mínimo

Pasos:
1. Carga de datos (train, validation y test)
2. Vectorización con TF-IDF (bigramas, sublinear_tf)
3. Entrenamiento de modelo baseline (LogisticRegression)
4. Evaluación en validación
5. Guardar artefactos (modelo + vectorizador)

In [4]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
import sys
import os

# Localizar src/classifier/ de forma robusta y ajustar cwd al directorio
# de este notebook para que rutas relativas (datasets/, data/, model/) funcionen
# independientemente de desde donde se lance Jupyter/VS Code.
_cwd = os.getcwd()
_candidates = [
    os.path.join(_cwd, "src", "classifier"),
    os.path.abspath(".."),
    os.path.abspath("."),
]
for _p in _candidates:
    if os.path.isfile(os.path.join(_p, "functions.py")):
        if _p not in sys.path:
            sys.path.insert(0, _p)
        # Cambiar cwd al directorio de este notebook
        os.chdir(os.path.join(_p, "classifier_dataset_artificial"))
        break

import functions  # noqa: E402
functions.MLFLOW_EXPERIMENT = "clasificador_riesgo_ia_artificial"
functions._DATASET_TAGS = {"dataset_type": "artificial", "dataset_source": "eu_ai_act_flagged"}

## 1. Carga de datos

In [6]:
import pandas as pd

train_df = pd.read_csv("data/processed/train.csv")
val_df = pd.read_csv("data/processed/validation.csv")
test_df = pd.read_csv("data/processed/test.csv")

X_train = train_df["text_final"]
y_train = train_df["etiqueta"]
X_val = val_df["text_final"]
y_val = val_df["etiqueta"]
X_test = test_df["text_final"]
y_test = test_df["etiqueta"]

print(f"Train: {len(X_train)} muestras")
print(f"Validation: {len(X_val)} muestras")
print(f"Test: {len(X_test)} muestras")

FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/train.csv'

## 2. Vectorización TF-IDF

In [ ]:
from functions import crear_tfidf

tfidf, X_train_tfidf, X_val_tfidf, X_test_tfidf = crear_tfidf(
    X_train, X_val, X_test,
    max_features=5000,
    ngram_range=(1, 2),
)

Vocabulario TF-IDF: 3773 términos
Forma train: (210, 3773)
Forma validation: (45, 3773)
Forma test: (45, 3773)


## 3. Entrenamiento del modelo baseline (LogisticRegression)

In [ ]:
from functions import entrenar_modelo_baseline

modelo = entrenar_modelo_baseline(X_train_tfidf, y_train, X_val_tfidf, y_val)

=== Resultados en VALIDACIÓN ===

                 precision    recall  f1-score   support

    alto_riesgo       0.81      0.93      0.87        14
    inaceptable       0.85      1.00      0.92        11
riesgo_limitado       1.00      0.90      0.95        10
  riesgo_minimo       1.00      0.70      0.82        10

       accuracy                           0.89        45
      macro avg       0.91      0.88      0.89        45
   weighted avg       0.90      0.89      0.89        45

F1-score macro (validación): 0.8886


## 4. Guardar artefactos

In [ ]:
import joblib
import os

output_dir = "model"
os.makedirs(output_dir, exist_ok=True)

joblib.dump(modelo, os.path.join(output_dir, "modelo_baseline.joblib"))
joblib.dump(tfidf, os.path.join(output_dir, "tfidf_vectorizer.joblib"))

print("Modelo baseline guardado en: model/modelo_baseline.joblib")
print("Vectorizador guardado en: model/tfidf_vectorizer.joblib")

Modelo baseline guardado en: model/modelo_baseline.joblib
Vectorizador guardado en: model/tfidf_vectorizer.joblib


In [ ]:
# ── MLflow (solo falla esta celda si el servidor no está disponible) ──
from functions import log_mlflow_safe
from sklearn.metrics import f1_score, accuracy_score

y_val_pred = modelo.predict(X_val_tfidf)

try:
    log_mlflow_safe(
        run_name="exp0_baseline",
        params={
            "model_type":         "LogisticRegression",
            "model_max_iter":     1000,
            "tfidf_max_features": 5000,
            "tfidf_ngram_range":  "(1, 2)",
            "tfidf_sublinear_tf": True,
        },
        metrics={
            "val_f1_macro": round(f1_score(y_val, y_val_pred, average="macro"), 4),
            "val_accuracy": round(accuracy_score(y_val, y_val_pred), 4),
        },
        artifacts=[
            "model/modelo_baseline.joblib",
            "model/tfidf_vectorizer.joblib",
        ],
        tags={"experimento": "0", "features": "tfidf"},
    )
    print("✓ Exp 0 (baseline) registrado en MLflow")
except Exception as e:
    print(f"⚠ MLflow no disponible: {e}")

Password obtenida desde variable de entorno local.
MLflow configurado correctamente → https://18.201.64.41/


c:\Users\rammu\anaconda3\envs\venv_proyecto\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '18.201.64.41'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\rammu\anaconda3\envs\venv_proyecto\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '18.201.64.41'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\rammu\anaconda3\envs\venv_proyecto\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host '18.201.64.41'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\r

✓ Run 'exp0_baseline' registrado en MLflow (https://18.201.64.41/)
✓ Exp 0 (baseline) registrado en MLflow
